# EmporiUm — Bonus Analysis
## Capstone 2 - Bonus Questions

**Analyst -** Hana Merawi     
**Territories -** Massachusetts & Maryland  
**Region -** Northeast  
**Data Period -** January 2022 – December 2025

## Setup - Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

In [25]:
store_sales = pd.read_csv('StoreSales.csv')
store_detail = pd.read_csv('StoreDetail.csv')
products = pd.read_csv('Products.csv')
product_categories = pd.read_csv('ProductCategories.csv')
customer_list = pd.read_csv('customer_list.csv', sep='|')

store_sales['Transaction Date'] = pd.to_datetime(store_sales['Transaction Date'])
store_sales['Month'] = store_sales['Transaction Date'].dt.to_period('M')
store_sales['Year'] = store_sales['Transaction Date'].dt.year

territory_stores = store_detail[store_detail['State'].isin(['Massachusetts', 'Maryland'])].copy()
territory_ids = territory_stores['Store ID'].tolist()
territory_sales = store_sales[store_sales['Store ID'].isin(territory_ids)].copy()
territory_sales['Month'] = territory_sales['Transaction Date'].dt.to_period('M')
territory_sales['Year'] = territory_sales['Transaction Date'].dt.year
sales_with_state = territory_sales.merge(territory_stores[['Store ID', 'State']], on='Store ID')

prod_with_cats = products.merge(product_categories, on=['CategoryID', 'SubcategoryID'])

print(f'Total records loaded - {len(store_sales):,}')
print(f'Territory records (MA + MD) - {len(territory_sales):,}')

Total records loaded - 335,129
Territory records (MA + MD) - 131,476


---
## Filtering, Sorting & Exploration

### Q1. January 2024 Transactions - Sorted Highest to Lowest. Which had sale amount greater than $500?

In [3]:
jan_2024 = store_sales[store_sales['Month'] == '2024-01'].copy()
jan_2024_sorted = jan_2024.sort_values('Sale Amount', ascending=False).reset_index(drop=True)

print(f'Total January 2024 transactions - {len(jan_2024_sorted):,}')
jan_2024_sorted.head(10)

Total January 2024 transactions - 6,654


,Transaction Date,Store ID,RewardsID,Prod Num,Sale Amount,Month,Year
0,2024-01-09,872,NaN,105341-IT,1763.7,2024-01,2024
1,2024-01-09,731,NaN,105341-IT,1763.7,2024-01,2024
2,2024-01-31,822,NaN,105341-IT,1763.7,2024-01,2024
3,2024-01-27,736,NaN,105341-IT,1763.7,2024-01,2024
4,2024-01-27,701,NaN,105341-IT,1763.7,2024-01,2024
5,2024-01-18,736,NaN,105341-IT,1763.7,2024-01,2024
6,2024-01-10,823,337.0,105341-IT,1763.7,2024-01,2024
7,2024-01-01,726,NaN,105341-IT,1763.7,2024-01,2024
8,2024-01-03,724,NaN,105326-IT,1717.7,2024-01,2024
9,2024-01-18,731,367.0,105326-IT,1717.7,2024-01,2024


In [4]:
jan_over_500 = jan_2024_sorted[jan_2024_sorted['Sale Amount'] > 500].reset_index(drop=True)

print(f'Transactions over $500 in January 2024: {len(jan_over_500):,}')
print(f'Highest sale ${jan_2024_sorted["Sale Amount"].max():,.2f}')
print(f'Total value of $500+ transactions ${jan_over_500["Sale Amount"].sum():,.2f}')
jan_over_500.head(20)

Transactions over $500 in January 2024: 497
Highest sale $1,763.70
Total value of $500+ transactions $521,679.69


,Transaction Date,Store ID,RewardsID,Prod Num,Sale Amount,Month,Year
0,2024-01-09,872,NaN,105341-IT,1763.7,2024-01,2024
1,2024-01-09,731,NaN,105341-IT,1763.7,2024-01,2024
2,2024-01-31,822,NaN,105341-IT,1763.7,2024-01,2024
3,2024-01-27,736,NaN,105341-IT,1763.7,2024-01,2024
4,2024-01-27,701,NaN,105341-IT,1763.7,2024-01,2024
5,2024-01-18,736,NaN,105341-IT,1763.7,2024-01,2024
6,2024-01-10,823,337.0,105341-IT,1763.7,2024-01,2024
7,2024-01-01,726,NaN,105341-IT,1763.7,2024-01,2024
8,2024-01-03,724,NaN,105326-IT,1717.7,2024-01,2024
9,2024-01-18,731,367.0,105326-IT,1717.7,2024-01,2024


**Results**
- Total January 2024 transactions 6,654
- Transactions over 500 - 497 
- Highest single sale was 1,763.70 - likely a laptop purchase
- These high value transactions are mostly Technology & Accessories products

### Q2. Find all products whose product number begins with 10525. What category and subcategory do they belong to?

In [5]:
prod_10525 = prod_with_cats[prod_with_cats['Prod Num'].str.startswith('10525')]

print(f'Products - starting with 10525: {len(prod_10525)}')
print(f'Category - {prod_10525["Category"].unique()}')
print(f'Subcategory - {prod_10525["Subcategory"].unique()}')
prod_10525[['Prod Num', 'Product', 'Category', 'Subcategory']].reset_index(drop=True)

Products - starting with 10525: 10
Category - ['Technology & Accessories']
Subcategory - ['Tablets']


,Prod Num,Product,Category,Subcategory
0,105250-IT,Realme Pad,Technology & Accessories,Tablets
1,105251-IT,Lenovo Tab P12 Pro,Technology & Accessories,Tablets
2,105252-IT,Microsoft Surface Pro 9,Technology & Accessories,Tablets
3,105253-IT,"Apple iPad Pro (12.9-inch, M2)",Technology & Accessories,Tablets
4,105254-IT,Chuwi HiPad Max,Technology & Accessories,Tablets
5,105255-IT,Lenovo Yoga Smart Tab,Technology & Accessories,Tablets
6,105256-IT,ASUS Vivobook 13 Slate OLED,Technology & Accessories,Tablets
7,105257-IT,Samsung Galaxy Tab S9 Ultra,Technology & Accessories,Tablets
8,105258-IT,Apple iPad Air (10.9-inch),Technology & Accessories,Tablets
9,105259-IT,Teclast T40 Pro,Technology & Accessories,Tablets


**Results:**
- There are 10 products starting with 10525
- All belong to Technology & Accessories - Tablets
- Includes Apple iPad, Samsung Galaxy Tab, Microsoft Surface Pro, and others

---
## Aggregation & NumPy

### Q4. What is total sales revenue, average transaction amount, and standard deviation across all in-store transactions?

In [21]:
total_revenue = store_sales['Sale Amount'].sum()
avg_transaction = np.mean(store_sales['Sale Amount'])
std_dev = np.std(store_sales['Sale Amount'])

print('Full Dataset Statistics')
print(f'Total Revenue:        ${total_revenue:>15,.2f}')
print(f'Average Transaction:  ${avg_transaction:>15,.2f}')
print(f'Standard Deviation:   ${std_dev:>15,.2f}')
print(f'Total Transactions:   {len(store_sales):>16,}')

Full Dataset Statistics
Total Revenue:        $  45,370,048.85
Average Transaction:  $         135.38
Standard Deviation:   $         279.72
Total Transactions:            335,129


**Result**
- Total revenue across all stores - 45,370,048.85
- Average transaction - $135.38
- Standard deviation of - 279.72 is more than double the average - meaning there is a very wide range from cheap stationery items under 5 to laptops over 1,700

### Q7 Top 10 best selling individual products by total revenue across the full dataset

In [10]:
prod_revenue = store_sales.merge(
    prod_with_cats[['Prod Num', 'Product', 'Category']], on='Prod Num', how='left'
)

top10_products = prod_revenue.groupby(['Prod Num', 'Product', 'Category'])['Sale Amount'].agg(
    Total_Revenue='sum',
    Units_Sold='count'
).reset_index().sort_values('Total_Revenue', ascending=False).head(10).reset_index(drop=True)

top10_products['Rank'] = top10_products.index + 1
top10_products['Total_Revenue'] = top10_products['Total_Revenue'].map('${:,.2f}'.format)
top10_products[['Rank', 'Product', 'Category', 'Total_Revenue', 'Units_Sold']]

,Rank,Product,Category,Total_Revenue,Units_Sold
0,1,MSI Creator Z16,Technology & Accessories,"$1,199,316.00",680
1,2,"Apple MacBook Pro (M2, 14-inch)",Technology & Accessories,"$1,082,151.00",630
2,3,"Apple MacBook Air (M2, 13-inch)",Technology & Accessories,"$1,003,136.80",584
3,4,HP Pavilion 14,Technology & Accessories,"$913,919.76",702
4,5,ASUS VivoBook S15,Technology & Accessories,"$900,524.00",697
5,6,Samsung Galaxy Book Pro,Technology & Accessories,"$890,257.12",698
6,7,Lenovo IdeaPad Flex 5,Technology & Accessories,"$877,535.40",678
7,8,LG Gram 17,Technology & Accessories,"$876,792.28",637
8,9,HP Spectre x360,Technology & Accessories,"$874,421.04",699
9,10,Acer Chromebook Spin 713,Technology & Accessories,"$872,358.20",674


**Results**
- All top 10 products are laptops from Technology & Accessories
- Number 1 is the MSI Creator Z16 at $1,199,316
- This confirms laptops are the single biggest revenue driver for EmporiUm

---
##      Merging & Grouping

### Q8. For each territory transaction display the date, sale amount, city, state, and territory manager

In [22]:
territory_detail = territory_sales.merge(
    store_detail[['Store ID', 'Store Location', 'State', 'Territory Manager']], on='Store ID'
)

q8_view = territory_detail[[
    'Transaction Date', 'Sale Amount', 'Store Location', 'State', 'Territory Manager'
]].copy().sort_values('Transaction Date').reset_index(drop=True)

print(f'Total territory transactions with full detail - {len(q8_view):,}')
q8_view.head(15)

Total territory transactions with full detail - 131,476


,Transaction Date,Sale Amount,Store Location,State,Territory Manager
0,2022-01-01,23.44,Annapolis,Maryland,Shruti Reddy
1,2022-01-01,25.45,North Harford,Maryland,Shruti Reddy
2,2022-01-01,20.36,North Harford,Maryland,Shruti Reddy
3,2022-01-01,9.45,North Harford,Maryland,Shruti Reddy
4,2022-01-01,57.70,North Harford,Maryland,Shruti Reddy
5,2022-01-01,6.02,North Harford,Maryland,Shruti Reddy
6,2022-01-01,61.14,North Harford,Maryland,Shruti Reddy
7,2022-01-01,13.13,North Harford,Maryland,Shruti Reddy
8,2022-01-01,25.45,North Harford,Maryland,Shruti Reddy
9,2022-01-01,20.36,North Harford,Maryland,Shruti Reddy


**Findings:**
- Successfully merged StoreSales with StoreDetail to show full transaction detail
- All 131,476 territory transactions now show city, state, and territory manager
- Massachusetts managed by Bo Heap, Maryland managed by Shruti Reddy

### Q9. Total revenue by region across the full dataset. Which region generates the most?

In [12]:
sales_with_region = store_sales.merge(store_detail[['Store ID', 'Region']], on='Store ID', how='left')

region_revenue = sales_with_region.groupby('Region')['Sale Amount'].agg(
    Total_Revenue='sum',
    Transactions='count',
    Avg_Transaction='mean'
).reset_index().sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

region_revenue['Pct_of_Total'] = (region_revenue['Total_Revenue'] / region_revenue['Total_Revenue'].sum() * 100).round(1).astype(str) + '%'
region_revenue['Total_Revenue'] = region_revenue['Total_Revenue'].map('${:,.2f}'.format)
region_revenue['Avg_Transaction'] = region_revenue['Avg_Transaction'].map('${:.2f}'.format)
region_revenue

,Region,Total_Revenue,Transactions,Avg_Transaction,Pct_of_Total
0,Northeast,"$24,237,526.98",182914,$132.51,53.4%
1,South,"$7,996,850.12",57300,$139.56,17.6%
2,East,"$6,723,039.53",48417,$138.86,14.8%
3,West,"$6,412,632.22",46498,$137.91,14.1%


**Results**
- Northeast generates the most revenue at 24,237,526 - over 53% of all EmporiUm revenue
- This is heavily influenced by Store 736 (North Harford, MD) at 8.7M
- South, East, and West regions each contribute between 6.4M and 8.0M

### Q12 Top 10 rewards members by total spend across the full dataset

In [15]:
all_rewards = store_sales[store_sales['RewardsID'].notna()].copy()
all_rewards['RewardsID'] = all_rewards['RewardsID'].astype(int)

top10_rewards = all_rewards.groupby('RewardsID')['Sale Amount'].agg(
    Total_Spend='sum',
    Transactions='count'
).reset_index().sort_values('Total_Spend', ascending=False).head(10).reset_index(drop=True)

top10_rewards = top10_rewards.merge(
    customer_list[['cust_id', 'name']], left_on='RewardsID', right_on='cust_id', how='left'
)
top10_rewards['Rank'] = top10_rewards.index + 1
top10_rewards['Total_Spend'] = top10_rewards['Total_Spend'].map('${:,.2f}'.format)
top10_rewards[['Rank', 'name', 'RewardsID', 'Total_Spend', 'Transactions']]

,Rank,name,RewardsID,Total_Spend,Transactions
0,1,Gloria Mendoza,245,"$17,534.76",79
1,2,Cole Brown,180,"$17,392.38",91
2,3,Ben Chang,74,"$17,320.93",80
3,4,Stanley H.,47,"$16,473.72",75
4,5,C.J. Cregg,375,"$16,466.96",81
5,6,Cal Abar,280,"$16,387.39",81
6,7,Lorna Morello,242,"$15,950.82",75
7,8,Gerri Kellman,409,"$15,798.41",80
8,9,Trent Lane,99,"$15,771.27",80
9,10,K. McClanahan,400,"$15,545.26",72


**Results**
- Top rewards member across all stores is Gloria Mendoza at 17,534 across 79 transactions
- Cole Brown from our Maryland territory ranks 2nd at 17,392 across 91 transactions
- Stanley H. from Massachusetts ranks 4th at 16,473 across 75 transactions
- Top rewards members average around 75-91 transactions - very loyal customers

### Q14 Identify the single best month and single worst month for each territory

In [24]:
monthly_rev = sales_with_state.groupby(['Month', 'State'])['Sale Amount'].sum().reset_index()
monthly_rev.columns = ['Month', 'State', 'Total Revenue']

for state in ['Massachusetts', 'Maryland']:
    data = monthly_rev[monthly_rev['State'] == state]
    best = data.loc[data['Total Revenue'].idxmax()]
    worst = data.loc[data['Total Revenue'].idxmin()]
    print(f' {state} ')
    print(f'  Best Month -  {str(best["Month"])}  — ${best["Total Revenue"]:>10,.2f}')
    print(f'  Worst Month - {str(worst["Month"])} — ${worst["Total Revenue"]:>10,.2f}')
    print()

 Massachusetts 
  Best Month -  2025-10  — $277,382.19
  Worst Month - 2022-09 — $ 58,716.74

 Maryland 
  Best Month -  2025-10  — $359,699.69
  Worst Month - 2022-09 — $158,952.74



**Results**
- Both territories share the same best month October 2025 - peak back to school and holiday season
- Massachusetts best: 277,382 - worst - 58,716 September 2022
- Maryland best: 359,699 - worst - 158,952 September 2022
- The gap between best and worst shows how important seasonal marketing is